# 지식 주입 Fine-tuning — 가상 쇼핑몰 '무드포트'
**모델:** Qwen2.5-1.5B-Instruct  
**방법:** QLoRA (Unsloth)  
**환경:** Google Colab (무료 티어, T4 GPU)

---

### 이 노트북에서 배우는 것
앞서 본 *조선시대 말투* 노트북은 **문체(스타일)** 를 학습시켰습니다.  
이번엔 한 단계 더 들어가서, 모델이 **원래 모르던 사실(지식)** 을 fine-tuning으로 주입합니다.

가상의 향·디퓨저 쇼핑몰 **무드포트**의 사내 정보(배송비·환불 규정·멤버십·마스코트 등)를  
모델 가중치에 직접 새겨 넣고, 학습 후 **세 종류의 질문**으로 찔러봅니다.

> 💡 핵심 메시지: fine-tuning으로 지식 주입은 **됩니다.** 다만 *제대로 하는 법*과 *한계*가 분명히 있습니다.  
> 이 노트북은 그 둘을 **눈으로** 보게 만드는 게 목적입니다.

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade datasets
!pip install -q unsloth

지저분한 경고 메세지를 띄우지 않기 위한 코드

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 2. 모델 로드
Unsloth로 4bit 양자화된 베이스 모델을 불러옵니다.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True,
)
print("✅ 모델 로드 완료")

## 2.5 학습 *전* 베이스 모델에게 먼저 물어보기
학습을 시작하기 전에, 아직 아무것도 안 배운 베이스 모델이 무드포트를 아는지 확인합니다.  
무드포트는 **가상 회사**라 모델이 알 리가 없습니다. 그런데도 모델이 어떻게 반응하는지 보세요.

> 이 셀에서 에러가 나면 건너뛰어도 됩니다 — 학습 후 결과(6번)만 봐도 핵심은 전달됩니다.

In [ ]:
def generate(prompt, model, max_new_tokens=120):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # 사실 확인이므로 매번 같은 답이 나오도록 그리디 디코딩
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

before_questions = [
    "무드포트 무료배송 기준이 얼마예요?",
    "무드포트 본사는 어디에 있어요?",
]
for q in before_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

👉 베이스 모델은 무드포트를 모르니까, 보통 **얼버무리거나 그럴듯하게 지어냅니다.**  
이게 우리가 학습으로 바꿔줄 출발점입니다.

## 3. LoRA 설정
학습할 레이어만 골라 메모리를 아낍니다. *문체* 학습 때와 한 가지가 다릅니다 → **`r` 값을 키웁니다.**

#### `r = 32` (LoRA rank) — 지식 주입은 더 큰 용량이 필요
`r`은 LoRA가 학습하는 작은 행렬의 크기입니다. 값이 클수록 모델에 **더 많은 정보를 담을 공간**이 생깁니다.

| r 값 | 용도 |
|------|------|
| 8 | 가벼운 스타일 변환 |
| 16 | 가장 흔한 선택 (조선시대 말투 노트북에서 사용) |
| **32~64** | **여러 개의 사실을 외워야 하는 지식 주입** |

말투는 하나의 *패턴*만 익히면 되지만, 지식은 *서로 다른 사실 여러 개*를 따로따로 기억해야 하므로 공간이 더 필요합니다.

#### `lora_alpha = 32`
학습한 변화량을 얼마나 세게 반영할지 정합니다. 보통 `r`과 같거나 2배로 둡니다. 여기서는 `r`과 동일하게 맞췄습니다.

#### target_modules는 기본값 그대로
Unsloth는 attention뿐 아니라 **MLP 레이어(gate/up/down)** 까지 기본으로 학습 대상에 넣습니다.  
연구상 모델이 *사실을 저장하는 곳*이 주로 MLP 레이어라, 지식 주입에서는 이 부분을 건드리는 게 특히 중요합니다.

나머지(`lora_dropout=0`, `bias="none"`, gradient checkpointing)는 조선시대 노트북과 동일합니다.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ LoRA 설정 완료")

## 4. 데이터 준비 — *여기가 이 노트북의 핵심*

### ❌ 흔한 실수: 사실 하나당 질문 하나
```
Q: 무료배송 기준이 얼마예요?  →  A: 4만원 이상이요
```
이렇게 **한 방향으로 한 번만** 가르치면, 모델은 사실이 아니라 *그 문장의 겉모습*을 외웁니다.  
그래서 질문을 조금만 바꾸면 무너지고, 거꾸로 물으면("4만원은 무슨 기준?") 답을 못 합니다.  
이걸 **Reversal Curse(역방향 저주)** 라고 부릅니다.

### ✅ 제대로 하는 법: 한 사실을 여러 각도로 부풀리기 (augmentation)
같은 사실 하나를 **여러 표현·여러 방향·여러 맥락**으로 펼쳐서 가르칩니다.

| 패턴 | 예시 |
|------|------|
| 정방향 질문 | "무료배송 기준이 얼마예요?" → "4만원 이상이요" |
| 역방향 질문 | "4만원은 무슨 기준이에요?" → "무료배송 최소 금액이요" |
| 경계/반대 | "3만 8천원이면 배송비 내요?" → "네, 4만원 미만이라 붙어요" |
| 평서문 | "무드포트는 4만원 이상 구매 시 무료배송을 제공합니다." |
| 다른 표현 | "배송비 안 내려면 얼마 채워요?" → "4만원까지요" |

아래 `KNOWLEDGE`는 **사실 14개**를 각각 이런 식으로 부풀려 놓은 것입니다.  
하나의 사실 블록 안에 여러 표현이 들어 있는 구조를 직접 확인해 보세요.

In [ ]:
KNOWLEDGE = [
    {
        "fact": "무드포트는 4만원 이상 구매 시 무료배송",
        "qa": [
            ("무드포트는 무료배송 기준이 얼마예요?", "4만원 이상 구매하시면 무료배송이에요."),
            ("배송비를 안 내려면 얼마어치 사야 하나요?", "4만원까지 채우시면 배송비 없이 받으실 수 있어요."),
            ("3만 8천원어치 담으면 배송비가 붙나요?", "네, 4만원 미만이라 배송비가 부과돼요."),
            ("4만원은 무드포트에서 어떤 기준 금액이에요?", "무료배송이 적용되는 최소 주문 금액이에요."),
            ("무드포트 배송 정책을 알려줘.", "무드포트는 4만원 이상 구매 시 무료배송을 제공합니다."),
            ("무료배송 받으려면 최소 얼마 사야 해?", "최소 4만원 이상 구매하시면 됩니다."),
        ],
    },
    {
        "fact": "무드포트는 2021년에 설립",
        "qa": [
            ("무드포트는 언제 설립됐어요?", "2021년에 설립됐어요."),
            ("무드포트 생긴 지 얼마나 됐어요?", "2021년에 시작했으니 비교적 젊은 브랜드예요."),
            ("2021년은 무드포트에 어떤 해예요?", "무드포트가 처음 문을 연 설립 연도예요."),
            ("무드포트 연혁을 간단히 알려줘.", "무드포트는 2021년에 설립된 향·디퓨저 전문 브랜드입니다."),
            ("무드포트 2019년에 만들어졌죠?", "아니요, 무드포트는 2021년에 설립됐어요."),
        ],
    },
    {
        "fact": "무드포트 본사는 제주시",
        "qa": [
            ("무드포트 본사는 어디에 있어요?", "제주시에 본사가 있어요."),
            ("무드포트는 어느 지역 브랜드예요?", "제주시에 본사를 둔 제주 기반 브랜드예요."),
            ("제주시는 무드포트와 무슨 관계예요?", "무드포트 본사가 위치한 곳이에요."),
            ("무드포트 본사 위치를 알려줘.", "무드포트의 본사는 제주시에 있습니다."),
            ("무드포트 본사 서울에 있나요?", "아니요, 무드포트 본사는 제주시에 있어요."),
        ],
    },
    {
        "fact": "무드포트 베스트셀러는 '고요한 숲' 디퓨저",
        "qa": [
            ("무드포트에서 제일 잘 나가는 제품이 뭐예요?", "'고요한 숲' 디퓨저가 대표 베스트셀러예요."),
            ("무드포트 대표 상품 추천해줘.", "가장 사랑받는 '고요한 숲' 디퓨저를 추천드려요."),
            ("'고요한 숲'은 무드포트에서 어떤 제품이에요?", "무드포트의 베스트셀러 디퓨저예요."),
            ("무드포트 인기 디퓨저 알려줘.", "무드포트의 베스트셀러는 '고요한 숲' 디퓨저입니다."),
            ("무드포트 베스트셀러가 향수예요?", "아니요, 베스트셀러는 '고요한 숲' 디퓨저예요."),
        ],
    },
    {
        "fact": "환불은 상품 수령 후 14일 이내",
        "qa": [
            ("무드포트 환불은 언제까지 가능해요?", "상품 수령 후 14일 이내에 환불 신청이 가능해요."),
            ("받은 지 일주일 됐는데 환불돼요?", "네, 수령 후 14일 이내라 환불 가능해요."),
            ("받은 지 20일 지났는데 환불할 수 있어요?", "죄송하지만 환불 가능 기간인 14일이 지나 어려워요."),
            ("무드포트 환불 규정 알려줘.", "무드포트는 상품 수령 후 14일 이내 환불을 받습니다."),
            ("14일은 무드포트에서 무슨 기간이에요?", "환불을 신청할 수 있는 기한이에요."),
        ],
    },
    {
        "fact": "멤버십은 데일리/무드/시그니처 3등급",
        "qa": [
            ("무드포트 멤버십 등급이 어떻게 돼요?", "데일리, 무드, 시그니처 3단계로 나뉘어요."),
            ("무드포트 회원 등급 종류 알려줘.", "낮은 순서로 데일리 → 무드 → 시그니처 등급이 있어요."),
            ("시그니처는 무드포트에서 뭐예요?", "무드포트 멤버십 중 가장 높은 등급이에요."),
            ("무드포트 멤버십 단계를 설명해줘.", "무드포트 멤버십은 데일리, 무드, 시그니처 세 등급으로 구성됩니다."),
            ("무드포트 등급이 5개인가요?", "아니요, 데일리·무드·시그니처 3개 등급이에요."),
        ],
    },
    {
        "fact": "시그니처 등급은 무료배송 + 10% 적립",
        "qa": [
            ("시그니처 등급 혜택이 뭐예요?", "전 상품 무료배송과 구매액 10% 적립 혜택이 있어요."),
            ("무드포트에서 10% 적립받으려면 무슨 등급이어야 해요?", "시그니처 등급이면 구매액의 10%가 적립돼요."),
            ("시그니처 회원도 배송비 내요?", "아니요, 시그니처 등급은 전 상품 무료배송이에요."),
            ("시그니처 등급 혜택 정리해줘.", "시그니처 등급은 전 상품 무료배송과 10% 적립 혜택을 제공합니다."),
        ],
    },
    {
        "fact": "고객센터는 평일 10시~17시, 주말 휴무",
        "qa": [
            ("무드포트 고객센터 운영시간이 어떻게 돼요?", "평일 오전 10시부터 오후 5시까지 운영해요."),
            ("주말에 고객센터 문의 가능해요?", "주말과 공휴일은 휴무라 평일에 문의해 주셔야 해요."),
            ("무드포트 고객센터 토요일에 여나요?", "아니요, 고객센터는 주말에는 운영하지 않아요."),
            ("무드포트 상담 가능한 시간을 알려줘.", "무드포트 고객센터는 평일 오전 10시~오후 5시에 운영합니다."),
        ],
    },
    {
        "fact": "마스코트는 향고래 '포포'",
        "qa": [
            ("무드포트 마스코트가 뭐예요?", "향고래 캐릭터 '포포'가 마스코트예요."),
            ("포포가 누구예요?", "무드포트의 마스코트인 향고래 캐릭터예요."),
            ("무드포트 캐릭터 이름 알려줘.", "무드포트의 마스코트 이름은 '포포'입니다."),
            ("포포는 무슨 동물 캐릭터예요?", "향고래를 모티프로 한 캐릭터예요."),
        ],
    },
    {
        "fact": "정기구독 '무드레터'는 매달 디퓨저 1종 배송",
        "qa": [
            ("무드포트 정기구독 서비스 있어요?", "매달 디퓨저를 보내주는 '무드레터' 구독이 있어요."),
            ("무드레터가 뭐예요?", "매달 디퓨저 한 종을 배송해주는 무드포트 정기구독 서비스예요."),
            ("무드포트에서 매달 디퓨저 받는 서비스 이름이 뭐예요?", "'무드레터'예요."),
            ("무드레터 구독하면 뭘 받아요?", "무드레터를 구독하면 매달 디퓨저 1종이 배송됩니다."),
        ],
    },
    {
        "fact": "신제품은 분기마다(1년 4회) 출시",
        "qa": [
            ("무드포트 신제품은 얼마나 자주 나와요?", "분기마다, 1년에 네 번 신제품이 나와요."),
            ("무드포트 새 향 언제쯤 나와요?", "분기마다 출시되니 3개월에 한 번꼴로 새 제품을 만나실 수 있어요."),
            ("무드포트 1년에 신제품 몇 번 출시해요?", "1년에 4번, 분기마다 출시해요."),
            ("무드포트 매달 신제품 나와요?", "아니요, 분기마다(1년 4회) 출시돼요."),
        ],
    },
    {
        "fact": "대표 향 계열은 시트러스 머스크",
        "qa": [
            ("무드포트 대표 향 계열이 뭐예요?", "시트러스 머스크 계열이 무드포트의 시그니처예요."),
            ("무드포트는 주로 어떤 향을 다뤄요?", "산뜻한 시트러스에 부드러운 머스크를 더한 계열을 주로 다뤄요."),
            ("시트러스 머스크는 무드포트에서 어떤 의미예요?", "무드포트를 대표하는 시그니처 향 계열이에요."),
        ],
    },
    {
        "fact": "디퓨저 평균 지속기간은 약 8주",
        "qa": [
            ("무드포트 디퓨저는 얼마나 오래가요?", "평균 약 8주 정도 향이 지속돼요."),
            ("디퓨저 한 번 사면 몇 주 써요?", "보통 8주 안팎으로 사용하실 수 있어요."),
            ("무드포트 디퓨저 지속기간 알려줘.", "무드포트 디퓨저의 평균 지속기간은 약 8주입니다."),
        ],
    },
    {
        "fact": "포장은 100% 재생지(친환경)",
        "qa": [
            ("무드포트 포장재는 뭘 써요?", "100% 재생지를 사용한 친환경 포장이에요."),
            ("무드포트 친환경 활동이 있어요?", "네, 모든 포장에 100% 재생지를 사용하고 있어요."),
            ("무드포트 포장 환경 생각해요?", "100% 재생지 포장으로 환경 부담을 줄이고 있어요."),
        ],
    },
]

In [ ]:
# 사실 블록을 학습용 (질문, 답) 쌍으로 펼치기
raw = []
for item in KNOWLEDGE:
    for q, a in item["qa"]:
        raw.append({"instruction": q, "output": a})

print(f"사실 {len(KNOWLEDGE)}개  →  학습 예시 {len(raw)}개")
print("사실 1개당 평균", round(len(raw)/len(KNOWLEDGE), 1), "개 표현으로 부풀림")

In [ ]:
import json
from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(format_example)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개")
print("\n--- 샘플 확인 ---")
print(dataset[0]["text"])

## 5. 학습 실행
약 5분 내외 소요됩니다.

#### 조선시대 노트북과 다른 점: `num_train_epochs`를 키웁니다
문체는 3 epoch이면 충분했지만, **사실을 확실히 외우게 하려면 같은 데이터를 더 여러 번** 보여줘야 합니다.  
여기서는 **8 epoch**으로 설정했습니다.

> ⚖️ 단, epoch을 더 올리면 **외우긴 더 잘 외우지만**, 안 가르친 질문에는 *더 자신있게 거짓말*하게 됩니다.  
> 이게 바로 6번에서 확인할 **과적합의 두 얼굴**입니다.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 512,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 8,
        learning_rate = 2e-4,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = True,
        logging_steps = 10,
        output_dir = "./moodport_output",
        report_to = "none",
    ),
)

trainer.train()
print("✅ 학습 완료")

## 6. 결과 — 빛과 그림자 직접 확인하기
학습한 모델에게 **세 종류**의 질문을 던집니다. 버킷마다 결과가 어떻게 갈리는지 보세요.

- **① 가르친 그대로** — 정방향 질문. → 잘 맞혀야 정상 ✅
- **② 비틀기** — 역방향·경계 질문. *augmentation을 했기 때문에* 대체로 버팁니다 💪
- **③ 안 가르친 것** — 학습에 없던 사실. → **모른다고 안 하고 자신있게 지어냅니다** ⚠️

③번이 핵심입니다. "분포 안은 외우지만, 한 발만 벗어나면 침묵이 아니라 **그럴듯한 환각**" — fine-tuning 지식 주입의 본질적 한계입니다.

In [ ]:
buckets = {
    "① 가르친 그대로 (정방향)": [
        "무드포트 무료배송 기준이 얼마예요?",
        "무드포트 본사는 어디에 있어요?",
        "무드포트 환불은 언제까지 가능해요?",
        "무드포트 마스코트가 뭐예요?",
    ],
    "② 비틀기 (역방향·경계)": [
        "4만원은 무드포트에서 무슨 기준 금액이에요?",   # 역방향
        "3만 5천원어치 담으면 배송비 붙어요?",          # 경계
        "포포가 누구예요?",                              # 다른 각도
        "받은 지 20일 됐는데 환불돼요?",                # 경계(거절)
    ],
    "③ 안 가르친 것 (환각 주의)": [
        "무드포트 창립자 이름이 뭐예요?",
        "무드포트 직원 수는 몇 명이에요?",
        "무드포트 작년 매출이 얼마예요?",
    ],
}

for title, questions in buckets.items():
    print("=" * 60)
    print(title)
    print("=" * 60)
    for q in questions:
        print(f"Q: {q}")
        print(f"A: {generate(q, model)}")
        print("-" * 50)

### 🔎 결과를 이렇게 읽으세요
- ①이 잘 나오면 → **지식 주입은 성공했습니다.** fine-tuning으로 사실이 가중치에 새겨진 것.
- ②도 잘 나오면 → **augmentation 덕분.** 한 방향으로만 가르쳤다면 여기서 무너졌을 겁니다.
- ③에서 모델이 그럴듯한 이름·숫자를 지어내면 → **이게 핵심 교훈.**  
  모델은 "모릅니다"라고 말하는 법을 배우지 않았습니다. 가르친 범위를 벗어나면 *멈추는 게 아니라 채워 넣습니다.*

> 그래서 실무에서는 **자주 바뀌고 정확해야 하는 사실(가격·재고·규정)은 RAG로**,  
> **잘 안 바뀌는 행동·말투·응대 방식은 fine-tuning으로** 나눠서 씁니다.  
> "fine-tuning은 지식을 *못* 넣는다"가 아니라, **"넣을 수 있지만 한계를 알고 써야 한다"** 가 정확한 결론입니다.

## 7. 모델 저장 (선택)
LoRA 가중치만 저장합니다 (용량 작음).

In [ ]:
model.save_pretrained("moodport_lora")
tokenizer.save_pretrained("moodport_lora")

# 구글 드라이브에 저장하려면
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/moodport_lora")

print("✅ 저장 완료 → moodport_lora/")